In [1]:
import torch
from torch import nn
from torch.nn import functional as F

In [12]:
#hyperparameters
batch_size = 32
block_size = 8
max_iter = 5000
eval_interval = 300
learning_rate = 1e-3
device = 'cuda' if torch.cuda.is_available() else 'cpu'
eval_iter = 200
n_embd = 32
# ---------------------

torch.manual_seed(1337)

with open('../data/input.txt', 'r', encoding='utf-8') as f:
    text = f.read()
    
chars = sorted(list(set(text)))
vocab_size = len(chars)

#map functions
stoi = {ch:i for i, ch in enumerate(chars)}
itos = {i:ch for i, ch in enumerate(chars)}
encode = lambda s: [stoi[ch] for ch in s] #take a string and map to list of ints
decode = lambda l: "".join([itos[i] for i in l]) #take a list of ints and map to string

#split 
data = torch.tensor(encode(text), dtype = torch.long)
n = int(0.9 * len(data))
train = data[:n]
val   = data[n:]

#data load 
def get_batch(split:str):
    data = train if split == 'train' else val
    ix = torch.randint(len(data) - block_size, (batch_size, ))
    x = torch.stack([data[i : i + block_size] for i in ix])
    y = torch.stack([data[i + 1 : i + block_size + 1] for i in ix])
    x, y = x.to(device), y.to(device)
    return x, y

@torch.no_grad()
def estimate_loss():
    out = {}
    model.eval()
    for split in ['train', 'val']:
        losses = torch.zeros(eval_iter)
        for k in range(eval_iter):
            x, y = get_batch(split)
            logits, loss = model(x, y)
            losses[k] = loss.item()
        out[split] = losses.mean()
    model.train()
    return out 

class Head(nn.Module):
    "one head of self-attention"
    def __init__(self, head_size):
        super().__init__()
        self.key   = nn.Linear(n_embd, head_size, bias=False)
        self.query = nn.Linear(n_embd, head_size, bias=False)
        self.value = nn.Linear(n_embd, head_size, bias=False)
        self.register_buffer('tril', torch.tril(torch.ones(block_size, block_size))) #tril is not a parameter, put in buffer
    
    def forward(self, x):
        B, T, C = x.shape 
        k = self.key(x)     #BT C = head_size
        q = self.query(x)   #BT C = head_size
        
        #compute attention "affinities"
        wei = q @ k.transpose(-2, -1)  * C**-0.5  #BTT
        wei = wei.masked_fill(self.tril[:T, :T] == 0, float('-inf'))
        wei = F.softmax(wei, dim=-1)
                
        v = self.value(x)   #BT C = head_size
        out = wei @ v #BTT @  #BT C ->  #BT C = head_size
        
        return out 
        




In [13]:
class MultiHeadAttention(nn.Module):
    def __init__(self, num_heads, head_size):
        super().__init__()
        self.heads = nn.ModuleList([Head(head_size) for _ in range(num_heads)])
        
    def forward(self, x):
        # x BT C = n_embd -> h(x) BT C = head_size
        return torch.cat([h(x) for h in self.heads], dim=-1)

In [14]:
        
class BigramLanguageModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.token_embedding_table = nn.Embedding(vocab_size, n_embd) #number of embedding dimensions
        self.position_embedding_table = nn.Embedding(block_size, n_embd) # each position has its own embd vec
        self.sa_heads = MultiHeadAttention(4, n_embd//4) # 4 heads of 8-dim self attention
        self.lm_head = nn.Linear(n_embd, vocab_size)
        
    def forward(self, idx, targets=None):
        B, T = idx.shape

        tok_emb = self.token_embedding_table(idx) #(BTC) C = n_embd
        pos_emb = self.position_embedding_table(torch.arange(T, device=device)) #(TC)
        x = tok_emb + pos_emb #now the token holds token identities and position it occur
                
        x = self.sa_heads(x) #The thinking part from the previous tocken, not just self 
        
        logits = self.lm_head(x) #BTC C = vocab_size
        
        
        if targets is None:
            loss = None
        else:
            B,T,C = logits.shape
            logits = logits.view(B*T, C)
            targets = targets.view(B*T)
            loss = F.cross_entropy(logits, targets)
        
        return logits, loss
    
    @torch.no_grad()
    def generate(self, idx, max_new_tokens):
        #idx is BT
        for _ in range(max_new_tokens):
            idx_cond = idx[:, -block_size:] #BT = 8, since block size= 8 in the model, if don trunk the self(idx) cannot work
            logits, loss = self(idx_cond) #BTC
            logits = logits[:, -1, :] #BC
            probs = F.softmax(logits, dim=-1) #BC
            idx_next = torch.multinomial(probs, num_samples = 1) #B1
            idx = torch.cat([idx, idx_next], dim=1) #B T+1
        return idx
    

In [15]:
model = BigramLanguageModel()    
m = model.to(device)

optimizer = torch.optim.AdamW(model.parameters(), lr = learning_rate)

for iter in range(max_iter):
    if iter % eval_interval == 0:
        losses = estimate_loss()
        print(f"step {iter}: train loss {losses['train']:.4f}, val loss {losses['val']:.4f}")
        
    xb, yb = get_batch('train')        
    
    logits, loss = model(xb, yb)
    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    optimizer.step()
    

step 0: train loss 4.2227, val loss 4.2226
step 300: train loss 2.8402, val loss 2.8482
step 600: train loss 2.6071, val loss 2.6175
step 900: train loss 2.5209, val loss 2.5235
step 1200: train loss 2.4699, val loss 2.4689
step 1500: train loss 2.4201, val loss 2.4294
step 1800: train loss 2.3949, val loss 2.4017
step 2100: train loss 2.3656, val loss 2.3681
step 2400: train loss 2.3427, val loss 2.3595
step 2700: train loss 2.3246, val loss 2.3454
step 3000: train loss 2.3231, val loss 2.3249
step 3300: train loss 2.3080, val loss 2.3153
step 3600: train loss 2.2897, val loss 2.3091
step 3900: train loss 2.2832, val loss 2.3017
step 4200: train loss 2.2751, val loss 2.3063
step 4500: train loss 2.2613, val loss 2.2749
step 4800: train loss 2.2517, val loss 2.2729


In [18]:
context = torch.zeros((1,1), dtype = torch.long, device = device)
print(decode(m.generate(context, max_new_tokens=500)[0].tolist()))
            
            


Romer,
Asf, thes posterstm why willgle he dealdemall bulaplikle, whice re Itrike isf lif thoos faut! Rorind acgpe! woulld resd ounjer, to heing bof theam Efautit? bor with hod
O
We
TOMad, sies an: bes thile, of aliastin pon.

Bit refer efar ther
Ford lomy antit fak co.
3!
Do'dsh tiplam; cade, fur
Ante as cufot thim!
Foriant aff.
'
Wreslond o' tore of thee of E
Sulat is trowato mely corkne do will kun:
DoTnaurs;
Kor ing gard son;
Nou wald coucr the 'Hory wis gadind Hesens thers, my theerce pore.



In [ ]:
""" Multi-head(多头,你代码里的 4 头):
  x (B,T,32) ─┬→ Head1(head_size=8) → (B,T,8) ─┐
              ├→ Head2(head_size=8) → (B,T,8) ─┤
              ├→ Head3(head_size=8) → (B,T,8) ─┼→ cat → (B,T,32)
              └→ Head4(head_size=8) → (B,T,8) ─┘
  4 个头并行,每个 head_size=8(32÷4),各算各的 attention,最后 torch.cat 拼回 32 维。

  对应你的代码:
  self.sa_heads = MultiHeadAttention(4, n_embd//4)   # 4 个头,每个 8 维

  def forward(self, x):
      return torch.cat([h(x) for h in self.heads], dim=-1)   # 并行 + 拼接

  关键:不是"复制 4 份",而是"分工 4 份"

  你说"并行多个 attention"方向对,但要补一个重点:

  - 总维度不变:单头是 1×32,多头是 4×8,加起来还是 32。不是变成
  4×32=128。所以计算量差不多,不是简单堆 4 倍。
  - 每个头独立学自己的 Q/K/V:4 个头有 4 套不同的 key/query/value
  权重(随机初始化不同、训练后各不相同)。
  - 所以每个头会关注不同的"关系" ← 这才是多头的意义。

  为什么多头更好(直觉)

  一个头只能学一种关注模式。但语言里,一个 token 同时有多种关系要照顾:

  以 "the cat sat on it" 里的 "it" 为例:
  - 头1 可能学"指代关系" → 关注 "cat"(it 指谁)
  - 头2 可能学"语法/主谓关系" → 关注 "sat"(动作)
  - 头3 可能学"位置/邻近关系" → 关注前一个词
  - 头4 可能学"句子边界" → 关注标点、换行

  单头被迫把这些混在一个 attention 里,顾此失彼。多头让每个头专注一种,最后拼起来 →
  信息更丰富。 """